# Silver — Limpieza y transformación
**Proyecto:** AI Tech Landscape Pipeline  
**Descripción:** Limpia, estandariza tipos y prepara los 3 datasets para la capa Gold.

In [1]:
import pandas as pd
import re
import os

BRONZE_PATH = "../data/bronze/"
SILVER_PATH = "../data/silver/"
os.makedirs(SILVER_PATH, exist_ok=True)

df_stocks = pd.read_parquet(f"{BRONZE_PATH}raw_big_tech_stocks.parquet")
df_gpu    = pd.read_parquet(f"{BRONZE_PATH}raw_gpu_companies.parquet")
df_ai     = pd.read_parquet(f"{BRONZE_PATH}raw_ai_companies.parquet")
print("✓ Datasets cargados")

✓ Datasets cargados


## Big Tech Stocks — limpieza

In [2]:
df_stocks["Date"] = pd.to_datetime(df_stocks["Date"])
df_stocks.columns = df_stocks.columns.str.strip().str.lower().str.replace(" ", "_")
df_stocks = df_stocks.drop(columns=["source_file"])

nulos = df_stocks.isnull().sum().sum()
df_stocks = df_stocks.dropna()

print(f"Nulos eliminados : {nulos}")
print(f"Rango de fechas  : {df_stocks['date'].min()} → {df_stocks['date'].max()}")
print(f"Empresas         : {sorted(df_stocks['ticker'].unique())}")
print(f"Filas finales    : {len(df_stocks):,}")
df_stocks.head(3)

Nulos eliminados : 0
Rango de fechas  : 2010-01-04 00:00:00 → 2023-01-24 00:00:00
Empresas         : ['AAPL', 'ADBE', 'AMZN', 'CRM', 'CSCO', 'GOOGL', 'IBM', 'INTC', 'META', 'MSFT', 'NFLX', 'NVDA', 'ORCL', 'TSLA']
Filas finales    : 45,088


,date,open,high,low,close,adj_close,volume,ticker
0,2010-01-04,7.622500,7.660714,7.585000,7.643214,6.515213,493729600,AAPL
1,2010-01-05,7.664286,7.699643,7.616071,7.656429,6.526476,601904800,AAPL
2,2010-01-06,7.656429,7.686786,7.526786,7.534643,6.422664,552160000,AAPL


## GPU Companies — limpieza

In [3]:
df_gpu.columns = df_gpu.columns.str.strip().str.lower().str.replace(" ", "_")
df_gpu["date"] = pd.to_datetime(df_gpu["date"])
df_gpu = df_gpu.drop(columns=["source_file"])
df_gpu = df_gpu.dropna()
df_gpu["empresa"] = df_gpu["empresa"].str.upper().str.strip()

print(f"Empresas GPU     : {sorted(df_gpu['empresa'].unique())}")
print(f"Rango de fechas  : {df_gpu['date'].min()} → {df_gpu['date'].max()}")
print(f"Filas finales    : {len(df_gpu):,}")
df_gpu.head(3)

Empresas GPU     : ['AMD', 'ASUS', 'INTEL', 'MSI', 'NVIDIA']
Rango de fechas  : 1980-03-18 00:00:00 → 2024-04-05 00:00:00
Filas finales    : 35,282


,date,open,high,low,close,adj_close,volume,empresa
0,1980-03-18,0.0,3.125000,2.937500,3.031250,3.031250,727200.0,AMD
1,1980-03-19,0.0,3.083333,3.020833,3.041667,3.041667,295200.0,AMD
2,1980-03-20,0.0,3.062500,3.010417,3.010417,3.010417,159600.0,AMD


## AI Companies — limpieza

In [9]:
df_ai.columns = df_ai.columns.str.strip().str.lower().str.replace(" ", "_")

def limpiar_revenue(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).replace("$", "").replace(",", "").strip()
    if "billion" in valor.lower():
        return float(re.sub(r"[^\d.]", "", valor)) * 1_000_000_000
    elif "million" in valor.lower():
        return float(re.sub(r"[^\d.]", "", valor)) * 1_000_000
    else:
        try:
            return float(re.sub(r"[^\d.]", "", valor))
        except:
            return None

df_ai["revenue_usd"] = df_ai["annual_revenue"].apply(limpiar_revenue)
df_ai["glassdoor_score"] = pd.to_numeric(df_ai["glassdoor_score"], errors="coerce")
df_ai["founded"] = pd.to_numeric(df_ai["founded"], errors="coerce").astype("Int64")

print(f"Empresas AI      : {len(df_ai)}")
print(f"Revenue parseado : {df_ai['revenue_usd'].notna().sum()} de {len(df_ai)}")
print(f"Nulos en revenue : {df_ai['revenue_usd'].isna().sum()}")
df_ai[["company_name", "founded", "annual_revenue", "revenue_usd", "glassdoor_score"]].head(10)

Empresas AI      : 100
Revenue parseado : 100 de 100
Nulos en revenue : 0


,company_name,founded,annual_revenue,revenue_usd,glassdoor_score
0,Alibaba Cloud,2009,$479.5 million,4.795000e+08,3.7
1,DataRobot,2012,$338.2 million,3.382000e+08,3.7
2,Google,1998,$305.6 billion,3.056000e+11,4.4
3,Hugging Face,2016,$40 million,4.000000e+07,4.3
4,H2O.ai,2011,$69.2 million,6.920000e+07,3.1
5,Rasa,2016,$18.8 million,1.880000e+07,3.9
6,Replicate,2019,$1.2 million,1.200000e+06,NaN
7,ActiveCampaign,2003,$195 million,1.950000e+08,3.7
8,ClickUp,2017,$158.7 million,1.587000e+08,3.4
9,Freshworks,2010,$596.4 million,5.964000e+08,3.8


## Guardar Silver

In [10]:
df_stocks.to_parquet(f"{SILVER_PATH}silver_big_tech_stocks.parquet", index=False)
df_gpu.to_parquet(f"{SILVER_PATH}silver_gpu_companies.parquet",      index=False)
df_ai.to_parquet(f"{SILVER_PATH}silver_ai_companies.parquet",        index=False)

print("=" * 50)
print("SILVER COMPLETADO")
print("=" * 50)
print(f"  Big Tech Stocks : {len(df_stocks):,} filas — {df_stocks['ticker'].nunique()} empresas")
print(f"  GPU Companies   : {len(df_gpu):,} filas — {df_gpu['empresa'].nunique()} empresas")
print(f"  AI Companies    : {len(df_ai):,} filas")
print("\nDatos listos para capa Gold →")

SILVER COMPLETADO
  Big Tech Stocks : 45,088 filas — 14 empresas
  GPU Companies   : 35,282 filas — 5 empresas
  AI Companies    : 100 filas

Datos listos para capa Gold →
